> Copyright 2026 Google LLC.
>
> Licensed under the Apache License, Version 2.0 (the "License");
> you may not use this file except in compliance with the License.
> You may obtain a copy of the License at
>
>      http://www.apache.org/licenses/LICENSE-2.0
>
> Unless required by applicable law or agreed to in writing, software
> distributed under the License is distributed on an "AS-IS" BASIS,
> WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
> See the License for the specific language governing permissions and
> limitations under the License.

# WeatherNext 2 Demo

## Installation and Initialization

In [ ]:
# @title Upgrade packages (kernel needs to be restarted after running this cell).
%pip install -U importlib_metadata

In [ ]:
# @title Pip install repo and dependencies and reconfigure jax if running on TPU.
%pip install --upgrade https://github.com/google-deepmind/weathernext/archive/master.zip --ignore-requires-python
# This is required due to outdated jax and libtpu versions in Colab TPU images.
%pip uninstall -y libtpu libtpu-nightly
%pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
# @title Imports

import dataclasses
import datetime
import math
from google.colab import auth
from google.cloud import storage
from typing import Optional
import haiku as hk
from IPython.display import HTML
from IPython import display
import ipywidgets as widgets
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray
import pandas as pd

import xarray_jax
from weathernext.utils import rollout
from weathernext.utils import checkpoint
from weathernext.utils import data_utils
from weathernext.utils import xarray_tree
from weathernext.utils import fiddle_config_io
from weathernext.weathernext2 import fgn
from weathernext.cyclones import constants as cyclone_constants
from weathernext.cyclones import direct_tracker_6h_v1_config
from weathernext.cyclones import ibtracs_netcdf_to_csv


In [ ]:
# @title Plotting functions

def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4,
    cyclone_dfs: Optional[dict[str, pd.DataFrame]] = None,
    init_time: Optional[np.datetime64] = None,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

  # Derive per-frame valid times from the lead-time coordinate + init_time.
  if cyclone_dfs:
    lead_times = first_data.coords["time"].values[:max_steps]
    valid_times = pd.Timestamp(init_time) + pd.to_timedelta(lead_times)

  # Build lat/lon to pixel coordinate mappings from the first data array.
  lat_coords = first_data.coords["lat"].values
  lon_coords = first_data.coords["lon"].values
  lat_min, lat_max = lat_coords.min(), lat_coords.max()
  lon_min, lon_max = lon_coords.min(), lon_coords.max()
  n_lat = len(lat_coords)
  n_lon = len(lon_coords)

  def latlon_to_pixel(lats, lons):
    """Convert lat/lon arrays to pixel coordinates for imshow."""
    px = (lons - lon_min) / (lon_max - lon_min) * (n_lon - 1)
    py = (lats - lat_min) / (lat_max - lat_min) * (n_lat - 1)
    return px, py

  def get_cyclone_positions(df, valid_time):
    """Filter a cyclone DataFrame for a single valid time, return (lats, lons)."""
    frame_df = df[df["valid_time"] == valid_time]
    lons = frame_df["lon"].values % 360  # Wrap to [0, 360) to match grid.
    return frame_df["lat"].values, lons

  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

  images = []
  scatters = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

    if cyclone_dfs and title in cyclone_dfs:
      lats_0, lons_0 = get_cyclone_positions(
          cyclone_dfs[title], valid_times[0])
      px, py = latlon_to_pixel(lats_0, lons_0)
      sc = ax.scatter(
          px, py, marker="x", c="red", s=20, linewidths=0.8, zorder=5)
      scatters.append((title, sc))
    else:
      scatters.append((title, None))

  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))
    # Update cyclone scatter positions.
    for title, sc in scatters:
      if sc is not None:
        lats_f, lons_f = get_cyclone_positions(
            cyclone_dfs[title], valid_times[frame])
        offsets = (np.column_stack(latlon_to_pixel(lats_f, lons_f))
                   if len(lats_f) > 0 else np.empty((0, 2)))
        sc.set_offsets(offsets)

  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
  plt.close(figure.number)
  return HTML(ani.to_jshtml())


# Load the Data and initialize the model

In [ ]:
# @title Load the weights, params and data.

# These can be updated as per info in the README (but will require accelerators
# not freely available in Colab).
model_name = "WeatherNextCyclones_Mini"
split = "2024"
data_resolution = "1.0"
# Steps must be one of "01", "04", "12", "20", or "30"
steps = "20"

config_name = f"weathernext2/configs/{model_name}"
weights_path = f"weathernext2/params/{model_name}_<{split}.npz"
data_path = f"weathernext2/dataset/source-hres_forecast_init-2024-10-07 00:00:00_res-{data_resolution}_levels-13_steps-{steps}.nc"

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")

with gcs_bucket.blob(data_path).open("rb") as f:
  example_batch = xarray.load_dataset(f).compute()
config = fiddle_config_io.get_fiddle_config_by_name(config_name)
with gcs_bucket.blob(weights_path).open("rb") as f:
  ckpt = checkpoint.load(f, fgn.CheckPoint)

try:
  ! wget https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/netcdf/IBTrACS.ALL.v04r01.nc
  ds_ibtracs = xarray.open_dataset("IBTrACS.ALL.v04r01.nc", engine="h5netcdf").load()
except:
  print(f"Failed to download IBTrACS data. Cells requiring IBTrACS will fail. See https://www.ncei.noaa.gov/news/cloud-migration for more details on recent IBTrACS data access issues.")
  ds_ibtracs = None


In [ ]:
# @title Choose data to plot

plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=500,
    description="Level")
plot_example_robust = widgets.Checkbox(value=True, description="Robust")
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.sizes["time"], value=example_batch.sizes["time"],
    description="Max steps")

widgets.VBox([
    plot_example_variable,
    plot_example_level,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Plot example data

plot_size = 7

data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
              robust=plot_example_robust.value),
}
fig_title = plot_example_variable.value
if "level" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_example_robust.value)


In [ ]:
# @title Extract training and eval data
task_config = config.task

train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", "6h"), # Only 1AR training.
    **dataclasses.asdict(task_config))

eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", f"{(example_batch.sizes['time']-2)*6}h"), # All but 2 input frames.
    **dataclasses.asdict(task_config))

print("All Examples:  ", example_batch.dims.mapping)
print("Train Inputs:  ", train_inputs.dims.mapping)
print("Train Targets: ", train_targets.dims.mapping)
print("Train Forcings:", train_forcings.dims.mapping)
print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)


In [ ]:
# @title Build jitted functions.

backend = jax.default_backend()
transformer_kwargs = config.predictor_kwargs['noisy_function_kwargs'][
    'mesh_model_ctor'
].keywords['transformer_kwargs']
# If running on GPU, use alternative attention implementation. Note that this
# is slower and significantly more memory hungry. Inference with WN-mini
# can fit on a P100 GPU, but note that gradient computation will fail OOM.
if backend == 'gpu':
  transformer_kwargs['attention_type'] = 'triblockdiag_mha'
# Overriding the following block sizes enable running the model on a TPU v5e-1.
# Note that in general, when running on different TPU versions, optimal block
# sizes are likely be different, and thus some tuning is recommended.
elif backend == 'tpu':
  transformer_kwargs.update({
      'block_q': 128,
      'block_kv': 128,
      'block_kv_compute': 128,
      'block_q_dkv': 128,
      'block_kv_dkv': 128,
      'block_kv_dkv_compute': 128,
  })

config_inference = fgn.PredictorConfig(
    task=config.task,
    predictor_constructor=config.predictor_constructor,
    predictor_kwargs=config.predictor_kwargs,
    predictor_wrappers=config.predictor_wrappers[:-1],  # Remove ensemble wrapper
)


@hk.transform
def run_forward(inputs, targets_template, forcings):
  predictor = fgn.construct_predictor(config_inference)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


# With ensemble wrapper for training on multiple samples.
config_train = config


@hk.transform
def loss_fn(inputs, targets, forcings):
  predictor = fgn.construct_predictor(config_train)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))


def grads_fn(params, inputs, targets, forcings):
  def _aux(params, inputs, targets, forcings):
    loss, diagnostics = loss_fn.apply(
        params, jax.random.PRNGKey(0), inputs, targets, forcings
    )
    return loss, diagnostics
  (loss, diagnostics), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, inputs, targets, forcings)
  return loss, diagnostics, grads

loss_fn_jitted = jax.jit(loss_fn.apply)
grads_fn_jitted = jax.jit(grads_fn)
run_forward_jitted = jax.jit(
    lambda rng, i, t, f: run_forward.apply(ckpt.params, rng, i, t, f)
)
# We also produce a pmapped version for running inference in parallel.
run_forward_pmap = xarray_jax.pmap(run_forward_jitted, dim="sample")

In [ ]:
# @title Randomly initialize network parameters
# This cell demonstrates how to initialize random network weights.

rng = jax.random.PRNGKey(0)
init_params = jax.jit(run_forward.init)(
    rng,
    train_inputs,
    train_targets,
    train_forcings,
)

num_params = sum(x.size for x in jax.tree_util.tree_leaves(init_params))
print(f"Initialized {num_params:,} parameters:")
for module_name in init_params:
  num_module_params = sum(
      x.size for x in jax.tree_util.tree_leaves(init_params[module_name]))
  print(f"  {module_name}: {num_module_params:,} parameters")

# Run the model

The `chunked_prediction_generator_multiple_runs` iterates over forecast steps, where the 1 step forecast is jitted and samples are pmapped across the chips.
This allows us to make efficient use of all devices and parallelise generating an ensemble across them. We then combine the chunks at the end to form our final forecast.

Note that the `Autoregressive rollout` cell will take longer than the standard inference time to run when executed for the first time, as this will include code compilation time. This cost does not increase with the number of devices, it is a fixed-cost one time operation whose result can be reused across any number of devices.

In [ ]:
# The number of ensemble members should be a multiple of the number of devices.
print(f"Number of local devices {len(jax.local_devices())}")

In [ ]:
# @title Autoregressive rollout (loop in python)

print("Inputs:  ", eval_inputs.dims.mapping)
print("Targets: ", eval_targets.dims.mapping)
print("Forcings:", eval_forcings.dims.mapping)

num_ensemble_members = 8 # @param int
rng = jax.random.PRNGKey(0)
# We fold-in the ensemble member, this way the first N members should always
# match across different runs which use take the same inputs, regardless of
# total ensemble size.
rngs = np.stack(
    [jax.random.fold_in(rng, i) for i in range(num_ensemble_members)], axis=0)

chunks = []
for chunk in rollout.chunked_prediction_generator_multiple_runs(
    # Use pmapped version to parallelise across devices.
    predictor_fn=run_forward_pmap,
    rngs=rngs,
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings,
    num_steps_per_chunk = 1,
    num_samples = num_ensemble_members,
    pmap_devices=jax.local_devices()
    ):
    chunks.append(chunk)
predictions = xarray.combine_by_coords(chunks)

In [ ]:
# @title Process IBTrACS for tracking

if ds_ibtracs is None:
  raise ValueError("IBTrACS data not available. See `Load the weights, params and data` cell.")

init_time = example_batch.isel(batch=0, time=1).datetime.values
ds_ibtracs["time"] = ds_ibtracs["time"].dt.round("1s")
all_storms_df, initial_storms_df = ibtracs_netcdf_to_csv.prepare_ibtracs_storms_dfs(
    ibtracs_ds=ds_ibtracs,
    init_time=init_time,
)


In [ ]:
# @title Run the tracker
tracker_config = direct_tracker_6h_v1_config.get_config()
tracker = tracker_config.tracker_constructor(**tracker_config.tracker_kwargs)

predicted_ensemble_tracks = []
for sample_idx in predictions.sample:
  data_to_track = predictions.isel(batch=0).sel(sample=sample_idx)
  data_to_track = tracker.preprocess_gridded_ds(data_to_track)
  data_to_track = data_to_track.expand_dims(forecast_datetime=[init_time])
  data_to_track = data_to_track.assign_coords(
      lead_time_secs=data_to_track.time.astype("timedelta64[s]").astype(int)
  )
  data_to_track = data_to_track.assign_coords(
      date_time=data_to_track.forecast_datetime.data + data_to_track.time
  )
  data_to_track = data_to_track.as_numpy()

  predicted_tracks = tracker(
      gridded_ds=data_to_track,
      initial_storms_df=initial_storms_df,
      do_cyclogenesis=True,
  )

  predicted_tracks[cyclone_constants.LON] = (
      predicted_tracks[cyclone_constants.LON] % 360
  )
  predicted_ensemble_tracks.append(predicted_tracks)

In [ ]:
# @title Choose predictions to plot

plot_pred_variable = widgets.Dropdown(
    options=predictions.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_pred_level = widgets.Dropdown(
    options=predictions.coords["level"].values,
    value=500,
    description="Level")
plot_pred_robust = widgets.Checkbox(value=True, description="Robust")
plot_pred_max_steps = widgets.IntSlider(
    min=1,
    max=predictions.sizes["time"],
    value=predictions.sizes["time"],
    description="Max steps")
plot_pred_samples = widgets.IntSlider(
    min=1,
    max=num_ensemble_members,
    value=num_ensemble_members,
    description="Samples")

widgets.VBox([
    plot_pred_variable,
    plot_pred_level,
    plot_pred_robust,
    plot_pred_max_steps,
    plot_pred_samples,
    widgets.Label(value="Run the next cell to plot the predictions. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Plot prediction samples and diffs

plot_size = 5
plot_max_steps = min(predictions.sizes["time"], plot_pred_max_steps.value)

fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

for sample_idx in range(plot_pred_samples.value):
  data = {
      "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
      "Predictions": scale(select(predictions.isel(sample=sample_idx), plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
      "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps) -
                          select(predictions.isel(sample=sample_idx), plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                        robust=plot_pred_robust.value, center=0),
  }
  cyclone_dfs = {
      "Targets": all_storms_df,
      "Predictions": predicted_ensemble_tracks[sample_idx],
  }
  display.display(
      plot_data(data, fig_title + f", Sample {sample_idx}", plot_size,
                plot_pred_robust.value, cyclone_dfs=cyclone_dfs,
                init_time=init_time))


In [ ]:
# @title Plot ensemble mean and CRPS

def crps(targets, predictions, bias_corrected = True):
  if predictions.sizes.get("sample", 1) < 2:
    raise ValueError(
        "predictions must have dim 'sample' with size at least 2.")
  sum_dims = ["sample", "sample2"]
  preds2 = predictions.rename({"sample": "sample2"})
  num_samps = predictions.sizes["sample"]
  num_samps2 = (num_samps - 1) if bias_corrected else num_samps
  mean_abs_diff = np.abs(
      predictions - preds2).sum(
          dim=sum_dims, skipna=False) / (num_samps * num_samps2)
  mean_abs_err = np.abs(targets - predictions).sum(dim="sample", skipna=False) / num_samps
  return mean_abs_err - 0.5 * mean_abs_diff


plot_size = 5
plot_max_steps = min(predictions.sizes["time"], plot_pred_max_steps.value)

fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

data = {
    "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Ensemble Mean": scale(select(predictions.mean(dim=["sample"]), plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Ensemble CRPS": scale(crps((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                        select(predictions, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                      robust=plot_pred_robust.value, center=0),
}
display.display(plot_data(data, fig_title, plot_size, plot_pred_robust.value))

# Train the model

The following operations requires larger amounts of memory than running inference.

The first time executing the cell takes more time, as it includes the time to jit the function.

In [ ]:
# @title Loss computation
loss, diagnostics = loss_fn_jitted(
    ckpt.params,
    jax.random.PRNGKey(0),
    train_inputs,
    train_targets,
    train_forcings)
print("Loss:", float(loss))
print(diagnostics)

In [ ]:
# @title Gradient computation (may OOM depending on runtime)
loss, diagnostics, grads = grads_fn_jitted(
    ckpt.params,
    inputs=train_inputs,
    targets=train_targets,
    forcings=train_forcings)
mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])
print(f"Loss: {loss:.4f}, Mean |grad|: {mean_grad:.6f}")
